# Prompt 15: HCP Terraform — Governance, Security, and Advanced Features

**Terraform Associate (004) — Objective 8 (continued): HCP Terraform Governance and Security**

This notebook covers the governance and security layer of HCP Terraform: teams and permissions, Sentinel and OPA policy enforcement, health assessments, dynamic provider credentials, private registry, audit logging, and advanced organizational features.

---

## Topics Covered

1. Teams and Permissions
2. Policy Enforcement — Sentinel and OPA
3. Health Assessments and Drift Detection
4. Dynamic Provider Credentials
5. Private Registry
6. Audit Logging
7. Additional Advanced Features (Explorer, Change Requests)
8. Exam-Style Questions
9. Real-World Scenarios

---

## 1. Teams and Permissions

### Organization Structure

```
HCP Terraform Organization
  ├── Teams
  │     ├── Team: platform-engineers
  │     ├── Team: app-developers
  │     └── Team: security
  ├── Projects
  │     ├── Project: web-application
  │     └── Project: networking
  └── Workspaces
        ├── web-app-production  (assigned to team: platform-engineers)
        └── networking-prod     (assigned to team: platform-engineers + security)
```

### Organization-Level Roles

| Role | Description |
|---|---|
| **Owner** | Full access — manage teams, billing, SSO, all workspaces |
| **Member** | Access determined by team workspace permissions |

> Only **owners** can manage the organization itself. Being a member of a team grants permissions at the workspace level, not organization level.

### Workspace-Level Permissions

| Permission | Can Do |
|---|---|
| **Read** | View runs, state, variables (no sensitive values). Cannot trigger runs. |
| **Plan** | Read + trigger plans (cannot apply). |
| **Write** | Plan + apply runs, manage variables. Cannot manage workspace settings. |
| **Admin** | Write + manage workspace settings, delete workspace, manage team access. |

### Token Types

| Token Type | Scope | Use Case |
|---|---|---|
| **User token** | One user's permissions | Personal CLI use, local development |
| **Team token** | Team's permissions on all workspaces the team can access | CI/CD pipelines shared by a team |
| **Organization token** | Organization admin operations | Organization-wide API access, managing workspaces via API |

> **Exam tip**: For CI/CD pipelines, use a **team token** — it provides consistent, auditable access tied to team permissions rather than an individual user. If the user leaves, the pipeline keeps working.

---

## 2. Policy Enforcement — Sentinel and OPA

### What is Policy Enforcement?

Policy enforcement in HCP Terraform lets organizations define rules that ALL Terraform plans must pass before they can be applied. Policies act as automated guardrails.

**Common use cases**:
- Require all EC2 instances to use approved AMIs
- Prevent resources from being created in non-approved AWS regions
- Enforce tagging standards on all resources
- Block instance types that exceed a cost threshold

### Policy Engines

| Engine | Description |
|---|---|
| **Sentinel** | HashiCorp's proprietary policy-as-code framework. Used by HCP Terraform natively. Policies written in Sentinel language. |
| **OPA (Open Policy Agent)** | Open-source policy engine. Policies written in Rego language. Supported as an alternative in HCP Terraform. |

### Where Policies Run in the Run Lifecycle

```
Trigger → Plan → [POLICY CHECK HERE] → Apply Confirmation → Apply
                        ↑
              Policies evaluate the plan
              BEFORE apply is possible
```

### Sentinel Enforcement Levels

| Enforcement Level | Behavior | Can Be Overridden? |
|---|---|---|
| **Advisory** | Policy failure generates a warning. Run can continue. | N/A — just a warning |
| **Soft Mandatory** | Policy failure blocks the apply. An **owner** can override and allow the apply. | Yes — by org owner |
| **Hard Mandatory** | Policy failure blocks the apply permanently. **No one can override it.** | No — absolute block |

> **Critical exam point**: Hard mandatory policies **cannot be overridden by anyone** — not even organization owners. Soft mandatory can be overridden by owners. Advisory is just a warning that doesn't block.

### Policy Sets

A **policy set** is a collection of policies applied to:
- All workspaces in the organization, OR
- Specific workspaces, OR
- Specific projects

Policy sets can be connected to a VCS repository for version-controlled policies.

In [ ]:
# Sentinel policy examples

sentinel_advisory_policy = '''
# enforce-tags.sentinel — Advisory level policy
# Warns if an AWS instance is missing a required "Environment" tag

import "tfplan/v2" as tfplan

# Get all AWS instances from the plan
aws_instances = filter tfplan.resource_changes as _, rc {
  rc.type is "aws_instance" and
  (rc.change.actions contains "create" or rc.change.actions contains "update")
}

# Check each instance has an Environment tag
instances_have_env_tag = rule {
  all aws_instances as _, instance {
    instance.change.after.tags contains "Environment"
  }
}

# Advisory enforcement — just a warning, run can proceed
main = rule {
  instances_have_env_tag
}
'''

sentinel_hard_mandatory = '''
# restrict-regions.sentinel — Hard mandatory policy
# Blocks any plan that creates resources outside allowed AWS regions

import "tfplan/v2" as tfplan

allowed_regions = ["us-east-1", "us-west-2", "eu-west-1"]

# Check all resource changes are in allowed regions
all_resources_in_allowed_regions = rule {
  all tfplan.resource_changes as _, rc {
    rc.change.after.region in allowed_regions
  }
}

# Hard mandatory — no override possible
# Configured as "hard-mandatory" in the HCP Terraform policy set settings
main = rule {
  all_resources_in_allowed_regions
}
'''

policy_set_config = '''
# sentinel.hcl — policy set configuration file
# This file lives in the VCS repository alongside policy files

policy "enforce-tags" {
  source            = "./enforce-tags.sentinel"
  enforcement_level = "advisory"       # warn only
}

policy "restrict-regions" {
  source            = "./restrict-regions.sentinel"
  enforcement_level = "hard-mandatory"  # absolute block
}

policy "restrict-instance-types" {
  source            = "./restrict-instance-types.sentinel"
  enforcement_level = "soft-mandatory"  # owner can override
}
'''

print(sentinel_advisory_policy)
print(sentinel_hard_mandatory)
print(policy_set_config)

### OPA Policy (Rego) Example

```rego
# restrict-instance-type.rego — OPA policy
# Denies plans that use expensive instance types

package terraform.policies.restrict_instance_type

import future.keywords.in

allowed_instance_types := {"t3.micro", "t3.small", "t3.medium"}

deny[msg] {
  rc := input.resource_changes[_]
  rc.type == "aws_instance"
  rc.change.actions[_] == "create"
  not rc.change.after.instance_type in allowed_instance_types
  msg := sprintf(
    "Instance type '%v' is not allowed. Use one of: %v",
    [rc.change.after.instance_type, allowed_instance_types]
  )
}
```

> **Exam tip**: Both Sentinel and OPA policies run **after plan, before apply**. Sentinel is HashiCorp-native; OPA uses the open-source Rego language. For the exam, focus on enforcement levels (advisory/soft-mandatory/hard-mandatory) — these are frequently tested.

---

## 3. Health Assessments and Drift Detection

### What Are Health Assessments?

HCP Terraform **automatically runs health assessments on a schedule** for workspaces. Health assessments detect two types of issues:

| Assessment Type | What It Detects |
|---|---|
| **Drift detection** | Real infrastructure differs from Terraform state (resources changed outside Terraform) |
| **Continuous validation** | `check` blocks in configuration fail (assertions about infrastructure state) |

### How It Works

```
Scheduled Health Assessment (HCP Terraform runs automatically)
        ↓
Equivalent to: terraform plan -refresh-only
        ↓
Compares actual infrastructure ↔ Terraform state
        ↓
If drift found → workspace health = UNHEALTHY
If check blocks fail → workspace health = UNHEALTHY
If no issues → workspace health = HEALTHY
        ↓
Results shown in workspace health dashboard
Notifications sent (email, Slack, etc.)
```

### Key Points

- Health assessments are available in **HCP Terraform Plus tier and above**
- They run **automatically** — no manual intervention needed
- Can be configured per-workspace (enable/disable, schedule)
- Finding drift does NOT automatically apply corrections — it only **reports** the drift; a human must decide to reconcile

### `check` Block Integration

```hcl
# check blocks in configuration are evaluated during health assessments
check "website_is_accessible" {
  data "http" "app" {
    url = "https://${aws_instance.web.public_ip}"
  }

  assert {
    condition     = data.http.app.status_code == 200
    error_message = "Website is not returning HTTP 200"
  }
}
```

If this check fails during a health assessment, the workspace is marked unhealthy.

> **Exam tip**: In Community Edition, drift is detected only when you manually run `terraform plan`. In HCP Terraform, health assessments run drift detection **automatically on a schedule** — this is a key differentiator.

---

## 4. Dynamic Provider Credentials

### The Problem: Static Credentials

Traditional HCP Terraform workspace configuration stores long-lived cloud credentials as environment variables:

```
Workspace variables:
  AWS_ACCESS_KEY_ID     = AKIA...  (stored persistently)
  AWS_SECRET_ACCESS_KEY = ****     (stored persistently)
```

**Security risks**:
- Credentials are persistent and must be rotated manually
- Long-lived credentials are a larger attack surface
- If leaked, they work indefinitely until rotated

### The Solution: Dynamic Credentials (OIDC)

Dynamic provider credentials use **OIDC (OpenID Connect)** — HCP Terraform acts as an OIDC identity provider and issues short-lived tokens to each run.

```
Run starts in HCP Terraform
        ↓
HCP Terraform issues a short-lived OIDC token
        ↓
Terraform provider exchanges OIDC token for cloud credentials
(via AWS IAM Role, Azure Managed Identity, GCP Workload Identity)
        ↓
Run executes with temporary credentials
        ↓
Credentials expire automatically after the run
        ↓
No persistent credentials stored anywhere
```

### Supported Cloud Providers

| Cloud | Mechanism |
|---|---|
| **AWS** | IAM Role with OIDC federation — HCP Terraform assumes the role via STS |
| **Azure** | Workload Identity Federation — Azure AD application with federated credential |
| **GCP** | Workload Identity Federation — GCP service account with federated identity |
| **Vault** | HCP Terraform authenticated to Vault JWT auth method — Vault issues secrets dynamically |

### Benefits

| Benefit | Description |
|---|---|
| **No stored secrets** | No long-lived keys in workspace variable store |
| **Run-scoped** | Credentials are valid only for the duration of the run |
| **No rotation needed** | Short-lived tokens expire automatically |
| **Auditability** | Cloud provider audit logs show which HCP workspace assumed the role |

> **Exam tip**: Dynamic credentials are considered a security best practice. The exam may test whether you know that dynamic credentials use **OIDC** and are **scoped to a single run** — they expire after the run completes.

In [ ]:
# Dynamic credentials configuration examples

aws_dynamic_credentials = '''
# AWS Dynamic Credentials Setup

# Step 1: In AWS, create an IAM OIDC Identity Provider
# Provider URL: https://app.terraform.io
# Audience: aws.workload.identity

# Step 2: Create an IAM Role with trust policy for HCP Terraform
# trust-policy.json
{
  "Version": "2012-10-17",
  "Statement": [{
    "Effect": "Allow",
    "Principal": {
      "Federated": "arn:aws:iam::123456789012:oidc-provider/app.terraform.io"
    },
    "Action": "sts:AssumeRoleWithWebIdentity",
    "Condition": {
      "StringEquals": {
        "app.terraform.io:aud": "aws.workload.identity",
        "app.terraform.io:sub": "organization:acme-corp:workspace:web-app-production:run_phase:*"
      }
    }
  }]
}

# Step 3: Configure workspace environment variables (NOT secrets)
# TFC_AWS_PROVIDER_AUTH = true
# TFC_AWS_RUN_ROLE_ARN  = arn:aws:iam::123456789012:role/hcp-terraform-role

# Step 4: No credential configuration needed in Terraform code!
provider "aws" {
  region = "us-east-1"
  # Credentials automatically injected via OIDC
}
'''

vault_dynamic_credentials = '''
# Vault Dynamic Credentials Setup
# HCP Terraform authenticates to Vault, Vault issues AWS credentials dynamically

# Workspace environment variables:
# TFC_VAULT_PROVIDER_AUTH = true
# TFC_VAULT_ADDR          = https://vault.example.com
# TFC_VAULT_NAMESPACE     = admin  (Vault Enterprise)
# TFC_VAULT_RUN_ROLE      = terraform-role

# In Terraform code — provider gets credentials from Vault automatically:
provider "vault" {
  address = "https://vault.example.com"
  # Auth handled by dynamic credentials — no token needed here
}

# Then use Vault provider to get AWS credentials dynamically:
data "vault_aws_access_credentials" "creds" {
  backend = "aws"
  role    = "deploy-role"
}

provider "aws" {
  access_key = data.vault_aws_access_credentials.creds.access_key
  secret_key = data.vault_aws_access_credentials.creds.secret_key
  token      = data.vault_aws_access_credentials.creds.security_token
}
'''

print(aws_dynamic_credentials)
print(vault_dynamic_credentials)

---

## 5. Private Registry

### What Is the Private Registry?

HCP Terraform includes a **Private Registry** that lets organizations publish and share:

- **Private modules** — reusable infrastructure components visible only to your organization
- **Private providers** — custom or third-party providers not in the public registry

### Private vs. Public Registry

| Feature | Public Registry | Private Registry |
|---|---|---|
| Location | `registry.terraform.io` | `app.terraform.io/<org>` |
| Visibility | Everyone | Your organization only |
| Authentication | Not required | HCP Terraform token |
| Module source format | `namespace/module/provider` | `<hostname>/<org>/<module>/<provider>` |
| Publishing | Open to anyone | Organization admins |

### Publishing Modules to the Private Registry

1. Connect the module repository to HCP Terraform via VCS
2. Tag the repository with a semantic version (`v1.0.0`, `v1.2.3`)
3. HCP Terraform automatically imports the tagged version
4. Module appears in the private registry — discoverable within the organization

### Consuming Private Registry Modules

The **same module block syntax** is used — just with a different source hostname:

```hcl
# Public registry module (for comparison)
module "vpc" {
  source  = "terraform-aws-modules/vpc/aws"   # public registry
  version = "~> 5.0"
}

# Private registry module
module "vpc" {
  source  = "app.terraform.io/acme-corp/vpc/aws"  # private registry
  version = "~> 2.0"
}
```

### Key Points

- Module versions are determined by **Git tags** on the source repository
- Private registry modules support the same **version constraint operators** as public registry modules
- When running via CLI-driven workflow, `terraform login` must be done first so the CLI can authenticate to download private modules
- Private **providers** follow a similar pattern — useful for internal/custom providers

> **Exam tip**: Private registry modules use the same `source` and `version` syntax as public registry modules — the only difference is the hostname prefix (`app.terraform.io/<org>/`) vs. the implicit public registry.

In [ ]:
# Private Registry module consumption examples

private_registry_example = '''
# terraform.tf
terraform {
  required_version = ">= 1.1"

  cloud {
    organization = "acme-corp"
    workspaces {
      name = "web-app-production"
    }
  }
}

# main.tf
# Using a module from the private registry
module "networking" {
  # Private registry: app.terraform.io/<org>/<module>/<provider>
  source  = "app.terraform.io/acme-corp/networking/aws"
  version = "~> 3.0"  # version constraints work the same as public registry

  # Input variables
  vpc_cidr        = "10.0.0.0/16"
  environment     = "production"
  azs             = ["us-east-1a", "us-east-1b", "us-east-1c"]
}

module "app_cluster" {
  # Another private registry module
  source  = "app.terraform.io/acme-corp/eks-cluster/aws"
  version = "= 2.4.1"  # pinned to exact version

  cluster_name = "production"
  vpc_id       = module.networking.vpc_id
  subnet_ids   = module.networking.private_subnet_ids
}

# Publishing a module:
# 1. Repository naming convention: terraform-<provider>-<module>
#    Example: terraform-aws-networking
# 2. Tag with semantic version: git tag v3.0.0 && git push --tags
# 3. HCP Terraform detects the new tag and publishes the version
'''

print(private_registry_example)

---

## 6. Audit Logging

### What Is Audit Logging?

HCP Terraform maintains an **organization-level audit log** that records every significant action performed in the organization.

### What Is Logged

| Category | Examples |
|---|---|
| **Runs** | Who triggered a plan, who confirmed an apply, who discarded a run |
| **Variables** | Who created/updated/deleted workspace variables or variable sets |
| **Workspace management** | Workspace created, deleted, settings changed |
| **Team management** | Team created, user added/removed, permissions changed |
| **Policy** | Policy set applied, policy overridden (soft mandatory) |
| **Authentication** | Login events, token creation/revocation |
| **State** | State uploaded, state lock acquired/released |

### Audit Log Access

- Available via the HCP Terraform **API** (`GET /api/v2/organizations/{org}/audit-trail`)
- Available in the **HCP Terraform UI** under organization settings
- Can be **streamed** to external log management systems (Splunk, Datadog, etc.)
- Available in **HCP Terraform Plus tier and above**

> **Exam tip**: Audit logging is an **organization-level** feature — it tracks all actions across all workspaces in the organization, not just individual workspaces. This is critical for compliance and security audits.

---

## 7. Additional Advanced Features

### Explorer

**Explorer** provides organization-wide visibility into all resources managed by Terraform across all workspaces.

| Feature | Description |
|---|---|
| **Resource inventory** | See all resources across all workspaces in one view |
| **Workspace health** | Overview of which workspaces have drift, check failures, or run errors |
| **Filtering** | Filter by resource type, provider, tag, workspace, or health status |
| **Module usage** | See which modules are in use and their versions across the organization |

Use case: "How many EC2 instances are we running across all workspaces?" — Explorer answers this instantly.

### Change Requests (Structured Runs)

**Change requests** provide a structured approval workflow for infrastructure changes:

```
Developer submits change request
  (describes the infrastructure change needed)
        ↓
Reviewers evaluate and approve
        ↓
Approved change triggers a run in the target workspace
        ↓
Full audit trail of who requested, reviewed, and approved
```

This adds a **formal ITSM-like change management layer** on top of the standard run workflow.

### HCP Terraform Tier Summary

| Feature | Free | Plus | Enterprise |
|---|---|---|---|
| Remote state | ✓ | ✓ | ✓ |
| Remote execution | ✓ | ✓ | ✓ |
| Variable sets | ✓ | ✓ | ✓ |
| Team permissions | ✓ | ✓ | ✓ |
| Sentinel/OPA policies | ✓ (limited) | ✓ | ✓ |
| Health assessments | ✗ | ✓ | ✓ |
| Audit logging | ✗ | ✓ | ✓ |
| Dynamic credentials | ✓ | ✓ | ✓ |
| SSO / SAML | ✗ | ✗ | ✓ |
| Self-hosted (TFE) | ✗ | ✗ | ✓ |

---

## 8. Exam-Style Questions

### Question 1

A platform team has a Sentinel policy configured with `enforcement_level = "soft-mandatory"` that restricts EC2 instance types to a pre-approved list. A developer's plan is blocked because it requests a `c5.4xlarge` instance. A senior engineer argues the deployment is time-sensitive and wants to proceed. What can happen?

**A.** No one can override a soft-mandatory policy — the plan must be modified to use an approved instance type  
**B.** An organization **owner** can override the soft-mandatory policy and allow the apply to proceed  
**C.** The developer can override the policy from their own account since they authored the run  
**D.** A hard-mandatory policy can be overridden by organization owners; soft-mandatory cannot  

<details><summary>Answer</summary>

**B** is correct. **Soft-mandatory** policies block the apply but can be **overridden by an organization owner**. The owner can review the situation and manually override the policy to allow the apply.

**A** is wrong — that describes hard-mandatory behavior.

**C** is wrong — only **organization owners** can override soft-mandatory policies, not ordinary users or developers.

**D** has it backwards — **hard-mandatory** cannot be overridden by anyone; **soft-mandatory** can be overridden by organization owners.

</details>

---

### Question 2

An organization wants to eliminate the need to store long-lived AWS access keys in HCP Terraform workspace variables. Which feature should they implement?

**A.** Rotate AWS credentials more frequently and update them in variable sets  
**B.** Store credentials in HashiCorp Vault and reference them with the `vault` provider  
**C.** Use dynamic provider credentials via OIDC federation between HCP Terraform and AWS IAM  
**D.** Use Sentinel policies to audit credential usage and detect exposure  

<details><summary>Answer</summary>

**C** is correct. **Dynamic provider credentials** use **OIDC** to federate identity between HCP Terraform and AWS IAM. HCP Terraform issues a short-lived OIDC token per run; Terraform exchanges it for temporary AWS credentials via STS `AssumeRoleWithWebIdentity`. No long-lived keys are stored anywhere.

**A** is wrong — rotating more often reduces the exposure window but still uses long-lived credentials stored in HCP. Dynamic credentials eliminate stored credentials entirely.

**B** is also valid (Vault dynamic credentials is a supported approach) but **C** describes the primary, simpler OIDC-native solution. For the exam, direct OIDC federation with cloud providers is the featured answer unless Vault is explicitly mentioned.

**D** is wrong — Sentinel policies enforce rules but do not provide credential management capabilities.

</details>

---

### Question 3

A DevOps team notices that an EC2 instance managed by Terraform was manually resized (instance type changed) directly in the AWS console. They are using HCP Terraform with health assessments enabled. What will happen WITHOUT any manual action from the team?

**A.** HCP Terraform will automatically apply `terraform apply` to revert the instance to the size defined in configuration  
**B.** HCP Terraform will detect the drift during its scheduled health assessment and mark the workspace as unhealthy — but will NOT automatically reconcile it  
**C.** The drift will only be detected the next time a developer manually runs `terraform plan`  
**D.** Terraform state will automatically update to reflect the new instance type, so no drift will be detected  

<details><summary>Answer</summary>

**B** is correct. HCP Terraform health assessments **automatically detect drift** on a schedule. When drift is detected (the instance type in AWS differs from what's in state), the workspace is marked **unhealthy** and the team is notified. However, HCP Terraform does **not** automatically apply corrections — a human must decide to run `terraform apply` (or `apply -refresh-only`) to reconcile.

**A** is wrong — HCP Terraform never applies changes automatically due to drift detection. Auto-apply only applies after a plan that was explicitly triggered.

**C** is wrong — this would be the Community Edition behavior. With HCP Terraform health assessments, detection is automated on a schedule.

**D** is wrong — state does not auto-update when infrastructure is changed outside Terraform. Drift means the state and actual infrastructure diverge.

</details>

---

### Question 4

An organization wants to publish an internal VPC module for use across all their Terraform workspaces. The module should only be visible to the organization and should support version pinning. What is the correct approach?

**A.** Publish the module to the public Terraform Registry with a private flag  
**B.** Publish the module to the HCP Terraform Private Registry using a VCS repository with Git tags for versions  
**C.** Store the module in a shared Git repository and reference it with a `git::https://` source in all consuming configurations  
**D.** Create the module as a local path (`./modules/vpc`) and copy it to every workspace's repository  

<details><summary>Answer</summary>

**B** is correct. The **HCP Terraform Private Registry** is designed exactly for this use case. Connect the VPC module repository to HCP Terraform, tag releases with semantic versions (e.g., `v2.0.0`), and the module becomes available to all workspaces in the organization using the same `source` and `version` syntax as the public registry. Version pinning works the same way as with public registry modules.

**A** is wrong — the public registry does not support private modules. Any module published there is visible to everyone.

**C** works but does not provide the same version pinning support or discoverability as the private registry. Git source modules support `ref=` for version tags, but this is less clean and doesn't integrate with the HCP Terraform registry UI.

**D** is the worst option — it creates multiple copies that diverge over time, defeats the purpose of shared modules, and makes updates extremely difficult.

</details>

---

## 9. Real-World Scenarios

<details><summary>Scenario 1: Enforcing Tagging Standards with Sentinel Policy</summary>

**Situation**: A cloud governance team at a financial services company needs to ensure every AWS resource created through Terraform has three mandatory tags: `CostCenter`, `Environment`, and `Owner`. Without these tags, billing allocation is impossible and compliance audits fail. Currently tags are documented in a wiki but developers frequently forget them.

**Solution — Sentinel Policy**:

```python
# enforce-mandatory-tags.sentinel
import "tfplan/v2" as tfplan

required_tags = ["CostCenter", "Environment", "Owner"]

# Get all AWS resources being created or updated
aws_resources = filter tfplan.resource_changes as _, rc {
  rc.provider_name is "registry.terraform.io/hashicorp/aws" and
  (rc.change.actions contains "create" or rc.change.actions contains "update")
}

# Verify each required tag is present
resources_have_required_tags = rule {
  all aws_resources as _, resource {
    all required_tags as tag {
      resource.change.after.tags contains tag
    }
  }
}

main = rule {
  resources_have_required_tags
}
```

**Policy Set Configuration**:

```python
# sentinel.hcl
policy "enforce-mandatory-tags" {
  source            = "./enforce-mandatory-tags.sentinel"
  enforcement_level = "soft-mandatory"  # owners can override for legacy resources
}
```

**Developer Experience**:

```
$ terraform plan → triggers HCP Terraform run

Plan succeeded. Policy check failed:
  enforce-mandatory-tags: FAILED (soft-mandatory)
  Resource aws_instance.web is missing required tags: ["CostCenter", "Owner"]

An organization owner can override this policy to proceed.
```

**Expected Outcome**: Developers are blocked from applying untagged resources. Violations are surfaced during plan review — before any infrastructure is created. An owner can override in exceptional cases (legacy migrations). Over time, tag compliance approaches 100%.

**Exam Sub-Objective**: Understand Sentinel policy enforcement levels (soft-mandatory) and how policies integrate into the run lifecycle (Objective 8).

</details>

---

<details><summary>Scenario 2: Eliminating Static AWS Credentials with OIDC Dynamic Credentials</summary>

**Situation**: A security audit at a startup found that AWS access keys stored as HCP Terraform workspace variables are never rotated. Two former employees still have their personal AWS keys that were used for CI/CD. The security team wants to eliminate all stored credentials.

**Setup Steps**:

**1. Create AWS IAM OIDC Identity Provider**:

```bash
# Using AWS CLI
aws iam create-open-id-connect-provider \
  --url https://app.terraform.io \
  --client-id-list aws.workload.identity \
  --thumbprint-list <thumbprint>
```

**2. Create IAM Role with trust policy**:

```json
{
  "Version": "2012-10-17",
  "Statement": [{
    "Effect": "Allow",
    "Principal": {
      "Federated": "arn:aws:iam::123456789012:oidc-provider/app.terraform.io"
    },
    "Action": "sts:AssumeRoleWithWebIdentity",
    "Condition": {
      "StringLike": {
        "app.terraform.io:sub": "organization:startup-co:project:*:workspace:*:run_phase:*"
      }
    }
  }]
}
```

**3. Set workspace environment variables** (NOT secrets):

```
TFC_AWS_PROVIDER_AUTH = true
TFC_AWS_RUN_ROLE_ARN  = arn:aws:iam::123456789012:role/hcp-terraform-deployer
```

**4. Remove stored AWS keys from workspace variables**

**5. Terraform configuration** — no changes needed:

```hcl
provider "aws" {
  region = "us-east-1"
  # No credentials here — OIDC injects them automatically
}
```

**Expected Outcome**: No AWS credentials are stored anywhere. Each run gets a fresh temporary credential set from AWS STS, scoped to only that run. When the run ends, the credentials expire. Former employees' keys are no longer a risk.

**Exam Sub-Objective**: Dynamic provider credentials via OIDC eliminate long-lived stored credentials (Objective 8).

</details>

---

<details><summary>Scenario 3: Investigating Drift with Health Assessments</summary>

**Situation**: An operations team receives a Slack notification from HCP Terraform: "Workspace production-api-server is unhealthy — drift detected." A developer manually changed an RDS instance's storage size in the AWS Console after a database growth incident. The team needs to decide: keep the manual change or revert to the Terraform-defined value.

**Investigation**:

```
HCP Terraform → Workspace: production-api-server → Health

Drift detected:
  aws_db_instance.main
    allocated_storage: 100 (in state) → 250 (actual)
    Updated outside Terraform
```

**Option A — Keep the manual change (update state to match)**:

```bash
# Via HCP Terraform UI: click "Update state" or run refresh-only apply
# OR via CLI-driven workflow:
$ terraform apply -refresh-only
# Updates state to reflect actual: allocated_storage = 250
# Does NOT make any changes to the actual database
```

**Then update Terraform configuration to match**:

```hcl
resource "aws_db_instance" "main" {
  # Update to match the actual value and save drift permanently
  allocated_storage = 250  # was 100
  ...
}
```

**Option B — Revert to Terraform configuration (apply to fix)**:

```bash
$ terraform plan   # shows: allocated_storage 250 → 100
$ terraform apply  # reverts database storage to 100GB
# WARNING: downgrading RDS storage is not supported by AWS!
# In practice, this would be caught before apply
```

**Expected Outcome**: Team decides to keep the 250GB value (database can't be downsized anyway). They update the Terraform configuration and run `apply -refresh-only` to reconcile state. Workspace health returns to HEALTHY.

**Exam Sub-Objective**: Health assessments detect drift automatically; drift detection does not auto-correct — human decision required (Objective 8).

</details>

---

<details><summary>Scenario 4: Centralizing Internal Modules in the Private Registry</summary>

**Situation**: A platform team at a large company has written 8 internal Terraform modules (networking, EKS, RDS, S3, IAM, monitoring, alerting, DNS). Each module is in its own GitHub repository. Currently, 20 product teams reference these modules via `git::https://` source, all pinned to `main` branch. When the platform team releases a fix, teams are forced to update manually — many are still using broken code months later.

**Solution — Private Registry**:

**Step 1**: Connect each module repository to HCP Terraform Private Registry
- Repository naming: `terraform-aws-networking`, `terraform-aws-eks`, etc.
- HCP Terraform automatically watches for new tags

**Step 2**: Tag releases for each module

```bash
# In each module repository
git tag v2.0.0
git push origin v2.0.0
# HCP Terraform immediately publishes v2.0.0 to private registry
```

**Step 3**: Teams update their source references

```hcl
# BEFORE: git source (pinned to branch — unstable)
module "networking" {
  source = "git::https://github.com/acme-corp/terraform-aws-networking.git?ref=main"
}

# AFTER: private registry (version-pinned — stable)
module "networking" {
  source  = "app.terraform.io/acme-corp/networking/aws"
  version = "~> 2.0"  # compatible updates, no breaking changes
}
```

**Step 4**: When platform team releases a patch (v2.0.1):

```bash
git tag v2.0.1 && git push origin v2.0.1
# Teams using "~> 2.0" automatically get v2.0.1 on next terraform init -upgrade
# Teams pinned to "= 2.0.0" must explicitly update to adopt the fix
```

**Expected Outcome**: All 8 modules are discoverable in HCP Terraform's private registry with documentation, input/output documentation, and version history. Product teams use consistent version constraints and the platform team has a reliable release process.

**Exam Sub-Objective**: Private registry for organization-scoped module distribution with version pinning (Objective 8).

</details>

---

<details><summary>Scenario 5: Role-Based Access Control for a Multi-Team Organization</summary>

**Situation**: A company has three teams working with Terraform in HCP Terraform: platform engineers (deploy everything), application developers (deploy app layer only), and security auditors (read-only review). The current setup has everyone as organization owners — a departing employee accidentally deleted a production workspace.

**Solution — Team-Based Permissions**:

**Organization Structure After Fix**:

```
Organization: acme-corp
  Owners: [platform-lead, cto]  (only 2 people)

  Teams:
    platform-engineers:
      Workspace permissions: Admin on all production workspaces
      Workspace permissions: Admin on all staging workspaces

    app-developers:
      Workspace permissions: Write on staging workspaces
      Workspace permissions: Plan on production workspaces (can plan, not apply)

    security-auditors:
      Workspace permissions: Read on all workspaces
```

**CI/CD Pipeline Setup**:

```bash
# DO NOT use personal user tokens in CI/CD
# Create a team token for the ci-deploy team

# In HCP Terraform UI:
# Teams → ci-deploy → Team API Token → Generate token
# Grant ci-deploy: Write on production workspaces

# In CI/CD pipeline (GitHub Actions example):
# Store token as GitHub secret: TF_TOKEN_APP_TERRAFORM_IO
env:
  TF_TOKEN_app_terraform_io: ${{ secrets.TF_TOKEN_APP_TERRAFORM_IO }}
```

**What Each Team Can Do**:

| Action | Platform Eng | App Dev | Security Auditor |
|---|---|---|---|
| Trigger plan | ✓ | ✓ (staging only) | ✗ |
| Confirm apply | ✓ | ✓ (staging) / ✗ (prod) | ✗ |
| Manage variables | ✓ | ✓ (staging) | ✗ |
| View state | ✓ | ✓ | ✓ |
| Delete workspace | ✓ | ✗ | ✗ |
| Override soft policy | ✓ (owners only) | ✗ | ✗ |

**Expected Outcome**: The accidental deletion scenario is prevented — only platform-lead and CTO (owners) can delete workspaces. Developers can freely work in staging but require platform team involvement for production applies. Security auditors have full visibility without change authority. Team tokens (not user tokens) are used for CI/CD so the pipeline doesn't break when employees leave.

**Exam Sub-Objective**: Teams and workspace-level permissions (read/plan/write/admin); team tokens vs user tokens for CI/CD (Objective 8).

</details>

---

## Quick Reference Summary

### Policy Enforcement Levels

| Level | Blocks Apply? | Override Possible? | By Whom? |
|---|---|---|---|
| Advisory | No | N/A | N/A |
| Soft Mandatory | Yes | Yes | Org Owner |
| Hard Mandatory | Yes | **No** | Nobody |

### Workspace Permissions

| Permission | Read | Plan | Write | Admin |
|---|---|---|---|---|
| View runs/state | ✓ | ✓ | ✓ | ✓ |
| Trigger plans | ✗ | ✓ | ✓ | ✓ |
| Confirm applies | ✗ | ✗ | ✓ | ✓ |
| Manage variables | ✗ | ✗ | ✓ | ✓ |
| Delete workspace | ✗ | ✗ | ✗ | ✓ |

### Key Concepts

| Concept | Key Fact |
|---|---|
| Sentinel | HashiCorp-native; policies in Sentinel language |
| OPA | Open-source alternative; policies in Rego language |
| Health assessments | Auto drift detection on schedule; reports but does NOT auto-correct |
| Dynamic credentials | OIDC-based; run-scoped; no stored long-lived keys |
| Private registry | Org-only modules; version via Git tags; same syntax as public registry |
| Audit logging | Org-level; tracks all actions; available via API and UI |
| Team token | Use in CI/CD (not personal user tokens) |

### Run Lifecycle with Policies

```
Trigger → Plan → [Policy Check] → Confirm → Apply
                       ↓
              Advisory: warn, continue
              Soft-Mandatory: block (owner can override)
              Hard-Mandatory: block (no override possible)
```